# 01: Data Collection

**Goal:** Download bill metadata from Congress.gov API and speeches from Stanford Congressional Record, then merge them.

## Steps
1. Set up Congress.gov API key
2. Fetch bills for Congress 110–114 (2007–2016)
3. Filter to economic subjects
4. Load Stanford speeches
5. Merge bills + speeches
6. Clean and save

**Data saved to:** `data/processed/bills_speeches_merged.csv`

In [ ]:
import sys
import os
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from src import data_utils

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Get Congress.gov API Key

**Instructions:**
1. Go to https://api.congress.gov
2. Register for a free account
3. Copy your API key
4. Paste it below OR set environment variable: `export CONGRESS_API_KEY=<key>`

In [ ]:
# Option 1: Retrieve from environment
try:
    api_key = data_utils.get_congress_gov_api_key()
    print(f"✓ API key found from environment")
except ValueError as e:
    print(f"Error: {e}")
    print("\nPlease set the CONGRESS_API_KEY environment variable or continue below.")
    # For testing, you can manually set here:
    # api_key = "YOUR_API_KEY_HERE"
    api_key = None

In [ ]:
# Option 2: Manually set API key (if env var not available)
# Uncomment and fill in your key:
# api_key = "PUT_YOUR_API_KEY_HERE"

# Check if we have a key
if api_key:
    print(f"✓ Ready to fetch data with API key: {api_key[:10]}...")
else:
    print("⚠ No API key found. Please set CONGRESS_API_KEY environment variable or uncomment the line above.")

## Step 2: Fetch Bills from Congress.gov

Download bill metadata for Congress 110–114, filtered to economic subjects.

In [ ]:
# Fetch bills for Congress 110–114
congress_numbers = [110, 111, 112, 113, 114]
all_bills = []

if api_key:
    for congress in congress_numbers:
        try:
            bills_df = data_utils.fetch_bills_from_congress_gov(congress, api_key)
            all_bills.append(bills_df)
            print(f"  Congress {congress}: {len(bills_df)} economic bills")
        except Exception as e:
            print(f"Error fetching Congress {congress}: {e}")
    
    # Combine all
    bills_combined = pd.concat(all_bills, ignore_index=True)
    print(f"\nTotal: {len(bills_combined)} bills across Congress 110–114")
else:
    print("Skipping API fetch. Please set API key above.")

In [ ]:
# Preview
if 'bills_combined' in locals():
    print(bills_combined.head())
    print(f"\nColumns: {bills_combined.columns.tolist()}")
    print(f"\nPass rate: {bills_combined['passed'].mean():.1%}")

## Step 3: Load Stanford Congressional Speeches

Load speeches from JSON files downloaded from https://data.stanford.edu/congress_text

**Expected file structure:** `data/raw/congress_110_speeches.json`, `congress_111_speeches.json`, etc.

In [ ]:
# Check if Stanford speech files exist
import os

raw_dir = "../data/raw"
if os.path.exists(raw_dir):
    files = os.listdir(raw_dir)
    speech_files = [f for f in files if "speech" in f.lower() or "congress_" in f.lower()]
    print(f"Files in {raw_dir}: {speech_files}")
    if not speech_files:
        print(f"\n⚠ No speech files found in {raw_dir}")
        print("Please download from https://data.stanford.edu/congress_text and save to data/raw/")
else:
    print(f"Directory {raw_dir} not found.")

In [ ]:
# Attempt to load speeches
try:
    speeches_df = data_utils.fetch_stanford_speeches([110, 111, 112, 113, 114], cache_dir="../data/raw")
    if len(speeches_df) > 0:
        print(f"Loaded {len(speeches_df)} speeches")
        print(f"\nColumns: {speeches_df.columns.tolist()}")
        print(f"\nSample:")
        print(speeches_df.head())
    else:
        print("No speeches loaded. Ensure Stanford files are in data/raw/")
except Exception as e:
    print(f"Error loading speeches: {e}")
    speeches_df = None

## Step 4: Merge Bills and Speeches

Match bill_id to link each bill with its floor speeches.

In [ ]:
# Merge if we have both datasets
if 'bills_combined' in locals() and speeches_df is not None and len(speeches_df) > 0:
    merged_df = data_utils.merge_bills_and_speeches(bills_combined, speeches_df)
    print(f"Merged dataset: {len(merged_df)} bills with speeches")
    print(f"\nColumns: {merged_df.columns.tolist()}")
else:
    print("Cannot merge: missing bills_combined or speeches_df")
    merged_df = None

In [ ]:
# Preview merged data
if merged_df is not None:
    print(merged_df[['bill_id', 'title', 'passed', 'speeches_combined']].head())
    print(f"\nPass rate in merged data: {merged_df['passed'].mean():.1%}")

## Step 5: Clean Data

Remove duplicates, nulls, and add derived features.

In [ ]:
# Clean
if merged_df is not None:
    cleaned_df = data_utils.clean_bill_speeches(merged_df)
    print(f"Cleaned dataset: {len(cleaned_df)} bills")
    print(f"\nNew columns: {cleaned_df.columns.tolist()}")
else:
    print("Skipping cleaning: no merged data")
    cleaned_df = None

In [ ]:
# Summary statistics
if cleaned_df is not None:
    print("\n=== Summary Statistics ===")
    print(f"Number of bills: {len(cleaned_df)}")
    print(f"Pass rate: {cleaned_df['passed'].mean():.1%}")
    print(f"\nBills by Congress:")
    print(cleaned_df['congress'].value_counts().sort_index())
    print(f"\nSpeech length (chars):")
    print(cleaned_df['speech_length'].describe())
    print(f"\nNumber of speakers per bill:")
    print(cleaned_df['num_speakers'].describe())

## Step 6: Save Processed Data

Save merged and cleaned data for downstream notebooks.

In [ ]:
# Save to CSV
if cleaned_df is not None:
    output_path = "../data/processed/bills_speeches_merged.csv"
    data_utils.save_processed_data(cleaned_df, output_path)
    print(f"\n✓ Data saved to {output_path}")
    print(f"  Shape: {cleaned_df.shape}")
else:
    print("Cannot save: no cleaned data")

## Summary

✓ Downloaded bill metadata from Congress.gov API  
✓ Loaded speeches from Stanford Congressional Record  
✓ Merged bills with speeches  
✓ Cleaned and saved to `data/processed/bills_speeches_merged.csv`

**Next:** Run `02_eda_preprocessing.ipynb`

# 02: Exploratory Data Analysis & Preprocessing

**Goal:** Analyze the merged dataset, visualize class balance, and preprocess text.

## Steps
1. Load merged dataset
2. Exploratory analysis: class balance, text statistics
3. Filter to economic speeches
4. Clean and preprocess text
5. Generate visualizations

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src import data_utils, nlp_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load Data

In [ ]:
# Load merged dataset from 01_data_collection
df = data_utils.load_processed_data("../data/processed/bills_speeches_merged.csv")

print(f"Loaded {len(df)} bills")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst row:")
print(df.iloc[0])

## Step 2: Exploratory Analysis

In [ ]:
print("=== Dataset Overview ===")
print(f"Shape: {df.shape}")
print(f"\nPass rate: {df['passed'].mean():.1%}")
print(f"\nValue counts (passed):")
print(df['passed'].value_counts())
print(f"\nBills by Congress:")
print(df['congress'].value_counts().sort_index())

In [ ]:
# Text statistics
print("=== Text Statistics ===")
print(f"\nSpeech length (characters):")
print(df['speech_length'].describe())
print(f"\nWord count per bill:")
print(df['speech_word_count'].describe())
print(f"\nNumber of speakers:")
print(df['num_speakers'].describe())

## Step 3: Class Balance Visualization

In [ ]:
# Plot class balance
viz_utils.plot_class_balance(
    df['passed'].values,
    output_path="../results/figures/class_balance.png"
)

## Step 4: Filter to Economic Speeches

In [ ]:
# Filter to bills with economic keywords in speeches
df_economic = nlp_utils.filter_economic_speeches(
    df,
    text_column="speeches_combined"
)

print(f"After economic filter: {len(df_economic)} bills")
print(f"Pass rate: {df_economic['passed'].mean():.1%}")

## Step 5: Preprocess Text

In [ ]:
# Extract additional text features
df_economic = nlp_utils.extract_text_features(
    df_economic,
    text_column="speeches_combined"
)

print("Extracted text features:")
print(df_economic[['text_length', 'text_word_count', 'sentence_count', 'avg_word_length']].describe())

## Step 6: Save Preprocessed Data

In [ ]:
# Save for next notebooks
df_economic.to_csv("../data/processed/bills_speeches_preprocessed.csv", index=False)
print(f"Saved preprocessed data: {len(df_economic)} bills")

**Next:** Run `03_tfidf_baseline.ipynb`

# 03: TF-IDF & Baseline Logistic Regression (ENHANCED)

**Research Question:** Does raw text (without engineered features) predict bill passage?

**This notebook answers:** Can we beat random chance (50% accuracy) with just TF-IDF features?

## Version: Enhanced with Validation
- ✅ Data validation checks
- ✅ Feature engineering validation
- ✅ Model convergence checks
- ✅ Checkpoint saves
- ✅ Research question tracking

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

print("✓ Imports successful!")

## STEP 1: Data Validation

In [ ]:
print("="*60)
print("RESEARCH QUESTION: Does text alone predict bill passage?")
print("="*60)
print()

# Load preprocessed data
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
print(f"✓ Loaded {len(df)} bills")

# Data validations
assert len(df) >= 100, f"ERROR: Dataset too small ({len(df)} < 100)"
print(f"✓ Dataset size OK: {len(df)} bills")

assert "speeches_combined" in df.columns, "ERROR: speeches_combined column missing"
assert "passed" in df.columns, "ERROR: passed column missing"
print(f"✓ Required columns present")

# Check for nulls in key columns
null_count = df[["speeches_combined", "passed"]].isnull().sum().sum()
assert null_count == 0, f"ERROR: {null_count} null values in key columns"
print(f"✓ No null values in speeches or target")

# Check binary target
assert set(df["passed"].unique()) == {0, 1}, "ERROR: Target is not binary"
pass_rate = df["passed"].mean()
print(f"✓ Binary target confirmed (Pass rate: {pass_rate:.1%})")

# Check class balance
if 0.3 <= pass_rate <= 0.7:
    print(f"✓ Class balance reasonable (not extreme)")
else:
    print(f"⚠ Warning: Class imbalanced (pass rate = {pass_rate:.1%})")

X_text = df["speeches_combined"].values
y = df["passed"].values

## STEP 2: TF-IDF Feature Engineering & Validation

In [ ]:
print("\n" + "="*60)
print("FEATURE ENGINEERING: TF-IDF Vectorization")
print("="*60)

# Create TF-IDF features
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text,
    max_features=5000,
    min_df=5,
    max_df=0.95,
)

print(f"\n✓ TF-IDF matrix created:")
print(f"  Shape: {X_tfidf.shape[0]} documents × {X_tfidf.shape[1]} features")

# Feature validation
assert X_tfidf.shape[0] == len(y), "ERROR: Feature matrix size mismatch"
print(f"✓ Sample count matches target")

assert X_tfidf.shape[1] > 10, "ERROR: Too few features extracted"
print(f"✓ Sufficient features: {X_tfidf.shape[1]} terms")

# Check sparsity
sparsity = 1 - (X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]))
assert sparsity > 0.5, "ERROR: Matrix not sparse enough"
print(f"✓ Matrix sparsity: {sparsity:.1%} (expected for TF-IDF)")

# Check feature range
tfidf_min = X_tfidf.data.min() if X_tfidf.nnz > 0 else 0
tfidf_max = X_tfidf.data.max() if X_tfidf.nnz > 0 else 0
assert 0 <= tfidf_min < tfidf_max <= 1, f"ERROR: TF-IDF values out of range [{tfidf_min}, {tfidf_max}]"
print(f"✓ TF-IDF values in valid range: [{tfidf_min:.4f}, {tfidf_max:.4f}]")

print(f"\nFeature names (first 10): {feature_names[:10]}")

## STEP 3: Model Training & Convergence Check

In [ ]:
print("\n" + "="*60)
print("MODEL TRAINING: Baseline Logistic Regression")
print("="*60)

# Train baseline logistic regression
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    baseline_model = model_utils.train_logistic_regression(X_tfidf, y, random_state=SEED)
    
    if w:
        print(f"⚠ Warnings during training:")
        for warning in w:
            print(f"  - {warning.message}")
    else:
        print(f"✓ Model converged without warnings")

print(f"✓ Logistic Regression model trained")
print(f"  Coefficients shape: {baseline_model.coef_.shape}")
print(f"  Intercept: {baseline_model.intercept_[0]:.4f}")

## STEP 4: Cross-Validation Evaluation & Validation

In [ ]:
print("\n" + "="*60)
print("EVALUATION: 5-Fold Stratified Cross-Validation")
print("="*60)

evaluator = model_utils.ModelEvaluator(random_state=SEED)
results = evaluator.evaluate_classifier(
    baseline_model,
    X_tfidf,
    y,
    model_name="Logistic Regression (Baseline)",
    cv_splits=5,
)

print(f"\n✓ Cross-validation complete")

# Validate results
for key in ["accuracy_mean", "auc_roc_mean", "precision_mean", "recall_mean", "f1_mean"]:
    value = results[key]
    assert 0 <= value <= 1, f"ERROR: {key} = {value} out of range [0, 1]"
    print(f"✓ {key}: {value:.4f} (valid)")

# Check if better than random
random_baseline = 0.5
if results["accuracy_mean"] > random_baseline:
    print(f"✓ BEATS RANDOM: Accuracy {results['accuracy_mean']:.1%} > {random_baseline:.1%}")
else:
    print(f"⚠ Does NOT beat random: Accuracy {results['accuracy_mean']:.1%} ≤ {random_baseline:.1%}")
    print(f"  → Text alone may not predict passage")

# Check AUC-ROC
if results["auc_roc_mean"] > 0.6:
    print(f"✓ Reasonable discrimination (AUC-ROC {results['auc_roc_mean']:.3f} > 0.6)")
else:
    print(f"⚠ Poor discrimination (AUC-ROC {results['auc_roc_mean']:.3f} ≤ 0.6)")

# Check precision/recall balance
if abs(results["precision_mean"] - results["recall_mean"]) < 0.1:
    print(f"✓ Precision/recall balanced")
else:
    print(f"⚠ Precision/recall imbalanced (diff: {abs(results['precision_mean'] - results['recall_mean']):.2f})")

In [ ]:
# Display results table
print("\n" + "="*60)
print("BASELINE MODEL PERFORMANCE")
print("="*60)
results_df = pd.DataFrame([results])
print(results_df[["model", "accuracy_mean", "auc_roc_mean", "f1_mean", "accuracy_std"]].to_string(index=False))

# Save results
results_df.to_csv("../results/tables/logistic_cv_scores.csv", index=False)
print(f"\n✓ Results saved to results/tables/logistic_cv_scores.csv")

## STEP 5: Feature Analysis & Interpretation

In [ ]:
print("\n" + "="*60)
print("FEATURE ANALYSIS: Top TF-IDF Words")
print("="*60)

# Get top TF-IDF words for each class
top_passed = nlp_utils.get_top_tfidf_words(vectorizer, X_tfidf, y, class_label=1, top_n=20)
top_failed = nlp_utils.get_top_tfidf_words(vectorizer, X_tfidf, y, class_label=0, top_n=20)

print(f"\nTop-20 words in PASSED bills:")
for i, (word, score) in enumerate(top_passed, 1):
    print(f"  {i:2d}. {word:20s} (TF-IDF: {score:.4f})")

print(f"\nTop-20 words in FAILED bills:")
for i, (word, score) in enumerate(top_failed, 1):
    print(f"  {i:2d}. {word:20s} (TF-IDF: {score:.4f})")

In [ ]:
# Interpretation
print(f"\n" + "="*60)
print("INTERPRETATION FOR REPORT")
print("="*60)

passed_words = set([w for w, _ in top_passed])
failed_words = set([w for w, _ in top_failed])
overlap = passed_words & failed_words

print(f"\nWord overlap between passed and failed: {len(overlap)} words")
if overlap:
    print(f"Overlapping words: {', '.join(sorted(overlap))}")
    print(f"→ These words appear frequently in both passed and failed bills")

unique_passed = passed_words - failed_words
unique_failed = failed_words - passed_words

print(f"\nWords unique to PASSED bills: {len(unique_passed)}")
if unique_passed:
    print(f"Examples: {', '.join(sorted(list(unique_passed))[:5])}")

print(f"\nWords unique to FAILED bills: {len(unique_failed)}")
if unique_failed:
    print(f"Examples: {', '.join(sorted(list(unique_failed))[:5])}")

## STEP 6: Visualizations

In [ ]:
# Word clouds
viz_utils.plot_wordcloud(
    df[df["passed"] == 1]["speeches_combined"].values,
    output_path="../results/figures/tfidf_wordcloud_passed.png",
    title="Top Terms in PASSED Bills"
)
print("✓ Saved: tfidf_wordcloud_passed.png")

In [ ]:
viz_utils.plot_wordcloud(
    df[df["passed"] == 0]["speeches_combined"].values,
    output_path="../results/figures/tfidf_wordcloud_failed.png",
    title="Top Terms in FAILED Bills"
)
print("✓ Saved: tfidf_wordcloud_failed.png")

## STEP 7: Save Checkpoint for Next Notebooks

In [ ]:
# Save vectorizer and features for reuse in later notebooks
checkpoint = {
    "vectorizer": vectorizer,
    "feature_names": feature_names,
    "X_tfidf": X_tfidf,
    "y": y,
    "baseline_model": baseline_model,
    "baseline_results": results
}

checkpoint_path = "../results/checkpoint_03_tfidf.pkl"
with open(checkpoint_path, "wb") as f:
    pickle.dump(checkpoint, f)

print(f"✓ Checkpoint saved: {checkpoint_path}")

## Summary: Research Question Answer

**Question:** Does raw text (TF-IDF features) alone predict bill passage?

**Answer:**

In [ ]:
print(f"\n" + "="*60)
print("RESEARCH QUESTION ANSWER")
print("="*60)

if results["accuracy_mean"] > 0.55:
    print(f"✓ YES (weak evidence)")
    print(f"  Baseline model achieves {results['accuracy_mean']:.1%} accuracy")
    print(f"  This beats random chance (50%)")
    print(f"  But effect is small (only {(results['accuracy_mean'] - 0.5) * 100:.1f} percentage points above chance)")
elif results["accuracy_mean"] > 0.50:
    print(f"⚠ MARGINAL (barely)")
    print(f"  Baseline model achieves {results['accuracy_mean']:.1%} accuracy")
    print(f"  Just barely better than random chance")
else:
    print(f"✗ NO")
    print(f"  Baseline model achieves {results['accuracy_mean']:.1%} accuracy")
    print(f"  This does NOT beat random chance (50%)")
    print(f"  Text alone is not predictive of passage")

print(f"\nNext step: Notebook 04 will use regularization (LASSO/Ridge)")
print(f"           to identify which specific words are predictive")

# 04: LASSO & Ridge Regularization

**Goal:** Train LASSO (L1) and Ridge (L2) regularized logistic regression for feature selection and comparison.

## Key Deliverable
Top-20 LASSO-selected words with economic interpretation

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load Data & TF-IDF Features

In [ ]:
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
X_text = df['speeches_combined'].values
y = df['passed'].values

# Create TF-IDF features (same as notebook 03)
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text,
    max_features=5000,
    min_df=5,
    max_df=0.95,
)

print(f"Data loaded: {len(df)} bills, {X_tfidf.shape[1]} TF-IDF features")

## Step 2: Train LASSO (L1) Logistic Regression

In [ ]:
# Train LASSO with CV for lambda tuning
lasso_model = model_utils.train_lasso_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
print(f"✓ LASSO model trained")
print(f"Best C (inverse lambda): {lasso_model.C_[0]:.6f}")

## Step 3: Train Ridge (L2) Logistic Regression

In [ ]:
# Train Ridge with CV for lambda tuning
ridge_model = model_utils.train_ridge_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
print(f"✓ Ridge model trained")
print(f"Best C (inverse lambda): {ridge_model.C_[0]:.6f}")

## Step 4: Extract & Visualize Top LASSO Features

In [ ]:
# Get top-20 LASSO features
top_lasso = model_utils.get_top_features_lasso(lasso_model, feature_names, top_n=20)
print("\nTop-20 LASSO Features (by absolute coefficient):")
print(top_lasso)

# Save
top_lasso.to_csv("../results/tables/lasso_top_features.csv", index=False)
print("\nSaved to results/tables/lasso_top_features.csv")

In [ ]:
# Plot LASSO coefficients
viz_utils.plot_feature_coefficients(
    top_lasso,
    output_path="../results/figures/lasso_coefficients.png",
    title="LASSO Feature Coefficients (Top-20)",
    max_features=20
)

## Step 5: Model Comparison

In [ ]:
# Evaluate both models
evaluator = model_utils.ModelEvaluator(random_state=SEED)

lasso_results = evaluator.evaluate_classifier(
    lasso_model, X_tfidf, y, model_name="LASSO Logistic", cv_splits=5
)
ridge_results = evaluator.evaluate_classifier(
    ridge_model, X_tfidf, y, model_name="Ridge Logistic", cv_splits=5
)

# Compare
comparison = pd.DataFrame([lasso_results, ridge_results])
print("\n=== LASSO vs Ridge Comparison ===")
print(comparison[['model', 'accuracy_mean', 'auc_roc_mean', 'f1_mean']])

# Save
comparison.to_csv("../results/tables/lasso_ridge_comparison.csv", index=False)

**Next:** Run `05_random_forest.ipynb`

# 05: Random Forest Classifier

**Goal:** Train random forest for non-linear feature interactions and compare to linear models.

## Key Deliverable
Top-30 RF feature importances vs. LASSO agreement

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load Data

In [ ]:
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
X_text = df['speeches_combined'].values
y = df['passed'].values

# Create TF-IDF features
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text,
    max_features=5000,
    min_df=5,
    max_df=0.95,
)

print(f"Data loaded: {len(df)} bills, {X_tfidf.shape[1]} TF-IDF features")

## Step 2: Train Random Forest

In [ ]:
# Convert sparse TF-IDF to dense for Random Forest
X_dense = X_tfidf.toarray()

# Train Random Forest
rf_model = model_utils.train_random_forest(
    X_dense, y,
    n_estimators=200,
    max_depth=None,
    random_state=SEED
)

print("✓ Random Forest trained with 200 trees")

## Step 3: Extract Feature Importances

In [ ]:
# Get top-30 RF features
top_rf = model_utils.get_top_features_rf(rf_model, feature_names, top_n=30)
print("\nTop-30 Random Forest Features:")
print(top_rf)

# Save
top_rf.to_csv("../results/tables/rf_top_features.csv", index=False)
print("\nSaved to results/tables/rf_top_features.csv")

In [ ]:
# Plot RF importances
viz_utils.plot_feature_importance(
    top_rf,
    output_path="../results/figures/rf_importance.png",
    title="Random Forest Feature Importances (Top-30)",
    max_features=30
)

## Step 4: Model Evaluation

In [ ]:
# Evaluate with CV
evaluator = model_utils.ModelEvaluator(random_state=SEED)
rf_results = evaluator.evaluate_classifier(
    rf_model, X_dense, y, model_name="Random Forest", cv_splits=5
)

print("\n=== Random Forest Performance ===")
for key, value in rf_results.items():
    if not key == "model":
        print(f"{key}: {value:.4f}")

# Save
results_df = pd.DataFrame([rf_results])
results_df.to_csv("../results/tables/rf_cv_scores.csv", index=False)

## Step 5: LASSO vs RF Feature Comparison

In [ ]:
# Load LASSO features
lasso_features = pd.read_csv("../results/tables/lasso_top_features.csv")

# Compare top-20 features
lasso_top20 = set(lasso_features['feature'].head(20))
rf_top20 = set(top_rf['feature'].head(20))

overlap = lasso_top20 & rf_top20

print(f"\nLASSO top-20 features: {len(lasso_top20)}")
print(f"RF top-20 features: {len(rf_top20)}")
print(f"Overlap: {len(overlap)} features")
print(f"\nOverlapping features:")
for feat in sorted(overlap):
    print(f"  - {feat}")

**Next:** Run `06_bert_classifier.ipynb` (optional) or proceed to `07_results_comparison.ipynb`

# 06: BERT Fine-tuning (Optional)

**Goal:** Fine-tune BERT for semantic bill classification (optional for neural network component).

**Status:** OPTIONAL. Complete 07_results_comparison without this if time is limited.

## Method
- Model: `bert-base-uncased` or `distilbert-base-uncased` (faster)
- Training: 3 epochs, lr=2e-5, batch_size=16
- Input: Bill title + summary (truncated to 512 tokens)
- Evaluation: 5-fold CV equivalent using Trainer API

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import torch
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports successful!")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 1: Prepare Data

In [ ]:
# Load data
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")

# Use bill title + summary as input (shorter than full speech)
df['text_input'] = df['title'].fillna('') + ' ' + df['summary'].fillna('')

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    df['text_input'].values,
    df['passed'].values,
    test_size=0.2,
    random_state=SEED,
    stratify=df['passed'].values
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train pass rate: {y_train.mean():.1%}, Test pass rate: {y_test.mean():.1%}")

## Step 2: Tokenize & Create Datasets

In [ ]:
# Load tokenizer and model
model_name = "distilbert-base-uncased"  # Faster than bert-base
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"Loaded {model_name}")

In [ ]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Create HF datasets
train_dataset = Dataset.from_dict({
    'text': X_train,
    'label': y_train
})

test_dataset = Dataset.from_dict({
    'text': X_test,
    'label': y_test
})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"Tokenized {len(train_dataset)} training samples")

## Step 3: Fine-tune

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="../models/bert_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=SEED,
    disable_tqdm=False,
)

print("Training arguments set")

In [ ]:
# Define compute metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    auc = roc_auc_score(labels, predictions)
    return {'accuracy': acc, 'auc': auc}

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer initialized. Starting fine-tuning...")

In [ ]:
# Fine-tune (this may take 5-10 min depending on GPU)
trainer.train()
print("✓ Fine-tuning complete")

## Step 4: Evaluate

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_dataset)
print("\n=== BERT Test Results ===")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

# Save
results_df = pd.DataFrame([test_results])
results_df.to_csv("../results/tables/bert_scores.csv", index=False)

**Note:** This notebook is OPTIONAL. The 3 linear/tree models (logistic, LASSO/Ridge, RF) are sufficient for a strong final report. Skip to `07_results_comparison.ipynb` if time is limited.

# 07: Model Comparison & Visualizations

**Goal:** Compare all trained models and generate final visualizations for the report.

## Outputs
- `results/tables/model_comparison.csv` - Performance table
- `results/figures/roc_curves.png` - ROC curves overlay
- `results/figures/confusion_matrices.png` - Confusion matrices grid

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, roc_curve, auc
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load All Results

In [ ]:
# Load model CV scores from previous notebooks
logistic_results = pd.read_csv("../results/tables/logistic_cv_scores.csv")
lasso_ridge_results = pd.read_csv("../results/tables/lasso_ridge_comparison.csv")
rf_results = pd.read_csv("../results/tables/rf_cv_scores.csv")

# Combine into one comparison table
all_results = pd.concat([
    logistic_results,
    lasso_ridge_results,
    rf_results
], ignore_index=True)

print("\n=== Model Comparison ===")
print(all_results[['model', 'accuracy_mean', 'auc_roc_mean', 'f1_mean', 'precision_mean', 'recall_mean']])

In [ ]:
# Try to load BERT results if they exist (optional)
try:
    bert_results = pd.read_csv("../results/tables/bert_scores.csv")
    bert_results['model'] = 'BERT'
    # Reformat columns if needed
    if 'eval_accuracy' in bert_results.columns:
        bert_results['accuracy_mean'] = bert_results['eval_accuracy']
    all_results = pd.concat([all_results, bert_results], ignore_index=True)
    print("\n✓ Included BERT results")
except FileNotFoundError:
    print("\n(BERT results not found - optional notebook)")

In [ ]:
# Save final comparison table
all_results.to_csv("../results/tables/model_comparison.csv", index=False)
print("\nSaved to results/tables/model_comparison.csv")

## Step 2: Prepare Data for Visualizations

In [ ]:
# Load data and features
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
X_text = df['speeches_combined'].values
y = df['passed'].values

# TF-IDF
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text, max_features=5000, min_df=5, max_df=0.95
)
X_dense = X_tfidf.toarray()

print(f"Data loaded: {len(df)} bills")

## Step 3: Generate ROC Curves

In [ ]:
# Train models for ROC predictions
from sklearn.model_selection import cross_val_predict

# Logistic Regression
lr_model = model_utils.train_logistic_regression(X_tfidf, y, random_state=SEED)
lr_proba = cross_val_predict(
    lr_model, X_tfidf, y, cv=5, method="predict_proba"
)[:, 1]

# LASSO
lasso_model = model_utils.train_lasso_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
lasso_proba = cross_val_predict(
    lasso_model, X_tfidf, y, cv=5, method="predict_proba"
)[:, 1]

# Ridge
ridge_model = model_utils.train_ridge_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
ridge_proba = cross_val_predict(
    ridge_model, X_tfidf, y, cv=5, method="predict_proba"
)[:, 1]

# Random Forest
rf_model = model_utils.train_random_forest(X_dense, y, n_estimators=200, random_state=SEED)
rf_proba = cross_val_predict(
    rf_model, X_dense, y, cv=5, method="predict_proba"
)[:, 1]

print("✓ Models trained for ROC curves")

In [ ]:
# Plot ROC curves
models_data = [
    ('Logistic Regression', y, lr_proba),
    ('LASSO', y, lasso_proba),
    ('Ridge', y, ridge_proba),
    ('Random Forest', y, rf_proba),
]

viz_utils.plot_roc_curves(
    models_data,
    output_path="../results/figures/roc_curves.png"
)

## Step 4: Generate Confusion Matrices

In [ ]:
# Get predictions for confusion matrices
lr_pred = cross_val_predict(lr_model, X_tfidf, y, cv=5)
lasso_pred = cross_val_predict(lasso_model, X_tfidf, y, cv=5)
ridge_pred = cross_val_predict(ridge_model, X_tfidf, y, cv=5)
rf_pred = cross_val_predict(rf_model, X_dense, y, cv=5)

# Get confusion matrices
models_cm = [
    ('Logistic Regression', confusion_matrix(y, lr_pred)),
    ('LASSO', confusion_matrix(y, lasso_pred)),
    ('Ridge', confusion_matrix(y, ridge_pred)),
    ('Random Forest', confusion_matrix(y, rf_pred)),
]

# Plot
viz_utils.plot_confusion_matrices(
    models_cm,
    output_path="../results/figures/confusion_matrices.png"
)

## Step 5: Summary Statistics for Report

In [ ]:
# Print summary for report writing
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

summary_cols = ['model', 'accuracy_mean', 'auc_roc_mean', 'f1_mean']
summary_table = all_results[summary_cols].sort_values('auc_roc_mean', ascending=False)

print("\n" + summary_table.to_string(index=False))

# Best model
best_model = summary_table.iloc[0]
print(f"\n✓ Best model: {best_model['model']} (AUC-ROC: {best_model['auc_roc_mean']:.4f})")

## Step 6: Key Insights for Report

In [ ]:
# Load feature lists for interpretation
lasso_features = pd.read_csv("../results/tables/lasso_top_features.csv")
rf_features = pd.read_csv("../results/tables/rf_top_features.csv")

print("\n" + "="*60)
print("KEY FINDINGS FOR REPORT")
print("="*60)

print("\n1. LASSO FEATURES (Words predicting passage/failure):")
print("\n   Positive (predict PASSAGE):")
positive = lasso_features[lasso_features['coefficient'] > 0].head(5)
for _, row in positive.iterrows():
    print(f"     - {row['feature']}: {row['coefficient']:.4f}")

print("\n   Negative (predict FAILURE):")
negative = lasso_features[lasso_features['coefficient'] < 0].head(5)
for _, row in negative.iterrows():
    print(f"     - {row['feature']}: {row['coefficient']:.4f}")

print("\n2. RANDOM FOREST TOP-5 FEATURES:")
for _, row in rf_features.head(5).iterrows():
    print(f"   - {row['feature']}: {row['importance']:.4f}")

In [ ]:
print("\n" + "="*60)
print("NEXT: Write final_report.md in report/ folder")
print("="*60)
print("\nReport structure:")
print("  1. Research Question (1 paragraph)")
print("  2. Data & Methods (1 page)")
print("  3. Results (1 page)")
print("  4. Discussion & Interpretation (1-2 pages)")
print("  5. Limitations (1 paragraph)")
print("\nKey interpretation questions:")
print("  - Which words predict passage? Why?")
print("  - Which model performs best and why?")
print("  - What is the baseline model (null result)?")
print("  - What confounders might explain the results?")